# QM9 + ZINC. first look
Loading via `torch_molecule`, drawing molecules with RDKit, and printing the property arrays.

In [ ]:
!pip install -q uv
!uv pip install --system torch-molecule rdkit matplotlib

## QM9

In [ ]:
from torch_molecule.datasets import load_qm9
QM9_COLS = ['mu','alpha','homo','lumo','gap','r2','zpve',
            'u0','u298','h298','g298','cv',
            'u0_atom','u298_atom','h298_atom','g298_atom',
            'A','B','C']
qm9 = load_qm9(local_dir='./data', target_cols=QM9_COLS)
print(len(qm9.data), 'molecules')
print('property array:', qm9.target.shape, qm9.target.dtype)
qm9.data[:5]

SMILES is the string format chemists use. uppercase letters are atoms (C, N, O, F), lowercase means aromatic, `=` is a double bond, `()` is a branch, digits close rings. Let me draw a handful to see what these actually look like.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

mols = [Chem.MolFromSmiles(s) for s in qm9.data[:12]]
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 200),
                     legends=list(qm9.data[:12]))

Each molecule has 19 properties computed by DFT. energies, dipole, HOMO/LUMO, the gap, polarizability, etc. 

In [ ]:
PROPS = QM9_COLS  # same order as we loaded; pretty labels not needed for indexing
first = qm9.target[0]
for name, val in zip(PROPS, first):
    print(f'{name:10s} {val:>12.4f}')

In [ ]:
# the headline ones for the thesis. gap and dipole, summary across the dataset
import numpy as np
gap_col, mu_col = PROPS.index('gap'), PROPS.index('mu')
print(f'gap   mean={qm9.target[:, gap_col].mean():.3f}  '
      f'min={qm9.target[:, gap_col].min():.3f}  '
      f'max={qm9.target[:, gap_col].max():.3f}')
print(f'mu    mean={qm9.target[:, mu_col].mean():.3f}  '
      f'min={qm9.target[:, mu_col].min():.3f}  '
      f'max={qm9.target[:, mu_col].max():.3f}')

QM9 doesn't ship logP / QED. RDKit computes them straight from SMILES.

In [ ]:
from rdkit.Chem import Crippen, QED

for s in qm9.data[:5]:
    m = Chem.MolFromSmiles(s)
    print(f'{s:25s}  logP={Crippen.MolLogP(m):+.2f}  QED={QED.qed(m):.2f}')

## ZINC250k

Bigger and drug-like. Different shape of data. molecules have 10+ heavy atoms, often with multiple fused rings.

In [ ]:
from torch_molecule.datasets import load_zinc250k
zinc = load_zinc250k(local_dir='./data')
print(len(zinc.data), 'molecules')
print('shipped target:', zinc.target)
zinc.data[:5]

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in zinc.data[:6]]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(280, 240),
                     legends=[s[:30] + '...' for s in zinc.data[:6]])

ZINC250k's drug-likeness scores (logP, QED, SAS) aren't shipped by torch_molecule's loader, so compute logP and QED with RDKit straight from the SMILES.

In [ ]:
# torch_molecule ships ZINC250k unlabeled, so compute logP / QED with RDKit on a sample
import numpy as np
sample = zinc.data[:2000]
logp = np.array([Crippen.MolLogP(Chem.MolFromSmiles(s)) for s in sample])
qed  = np.array([QED.qed(Chem.MolFromSmiles(s)) for s in sample])
print(f'logP  mean={logp.mean():.2f}  min={logp.min():.2f}  max={logp.max():.2f}')
print(f'QED   mean={qed.mean():.2f}  min={qed.min():.2f}  max={qed.max():.2f}')

A few ZINC molecules with their RDKit-computed logP and QED.

In [ ]:
for s in zinc.data[:5]:
    m = Chem.MolFromSmiles(s)
    print(f'{s[:40]:40s}  logP={Crippen.MolLogP(m):+.2f}  QED={QED.qed(m):.2f}')

## Properties: what comes from `.target` vs what RDKit computes

Useful to keep this straight. `.target` is whatever the dataset shipped. for QM9 those are DFT-computed quantum scalars; for ZINC those are the three drug-likeness scores from the original paper. RDKit, on the other hand, computes structural / drug-like properties on the fly from any SMILES. no dataset needed.

In [ ]:
from rdkit.Chem import Descriptors, Crippen, QED, rdMolDescriptors

print('--- shipped in .target ---')
print(f'QM9   ({qm9.target.shape[1]} cols): {QM9_COLS}')
print(f'ZINC  (none shipped by torch_molecule; compute on the fly with RDKit)')

print()
print('--- what RDKit gives on any Mol ---')
m = Chem.MolFromSmiles(qm9.data[0])
rdkit_props = {
    'MolWt':       Descriptors.MolWt(m),
    'logP':        Crippen.MolLogP(m),
    'QED':         QED.qed(m),
    'TPSA':        rdMolDescriptors.CalcTPSA(m),
    'HBA':         rdMolDescriptors.CalcNumHBA(m),
    'HBD':         rdMolDescriptors.CalcNumHBD(m),
    'RotBonds':    rdMolDescriptors.CalcNumRotatableBonds(m),
    'RingCount':   rdMolDescriptors.CalcNumRings(m),
    'AromaticRings': rdMolDescriptors.CalcNumAromaticRings(m),
    'HeavyAtoms':  m.GetNumHeavyAtoms(),
    'Formula':     rdMolDescriptors.CalcMolFormula(m),
}
for k, v in rdkit_props.items():
    print(f'  {k:16s} = {v}')